# Data Science Assignment 02: Part 1 - Exploratory Data Analysis

## Titanic Dataset Analysis
This notebook focuses on understanding the Titanic dataset before drawing any charts. Good visualization begins with knowing what the data actually contains, where it is incomplete, and what each variable represents.

## Setup (Required)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Load dataset
df = sns.load_dataset('titanic')
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

Dataset loaded successfully!
Shape: (891, 15)


## Q1: Initial Inspection

### Dataset Overview

The Titanic dataset records the fate of 891 passengers aboard the RMS Titanic. This dataset contains 15 columns with various attributes about the passengers including demographics, ticket information, and survival outcomes.

### Q1(a): Display First 8 and Last 5 Rows

In [2]:
# Display first 8 rows
print("First 8 rows of the dataset:")
print(df.head(8))
print("\n" + "="*100 + "\n")

# Display last 5 rows
print("Last 5 rows of the dataset:")
print(df.tail(5))

First 8 rows of the dataset:
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   
5         0       3    male   NaN      0      0   8.4583        Q  Third   
6         0       1    male  54.0      0      0  51.8625        S  First   
7         0       3    male   2.0      3      1  21.0750        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        

### Q1(b): Shape, Data Types, and Statistical Summary

In [3]:
# Dataset shape
print("Dataset Shape:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\n" + "="*100 + "\n")

# Data types
print("Data Types:")
print(df.dtypes)
print("\n" + "="*100 + "\n")

# Statistical summary for numeric columns
print("Statistical Summary - Numeric Columns:")
print(df.describe())
print("\n" + "="*100 + "\n")

# Statistical summary for categorical-style columns
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
print("Statistical Summary - Categorical Columns:")
print(df[categorical_cols].describe())


Dataset Shape:
Rows: 891, Columns: 15


Data Types:
survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class          category
who                 str
adult_male         bool
deck           category
embark_town         str
alive               str
alone              bool
dtype: object


Statistical Summary - Numeric Columns:
         survived      pclass         age       sibsp       parch        fare
count  891.000000  891.000000  714.000000  891.000000  891.000000  891.000000
mean     0.383838    2.308642   29.699118    0.523008    0.381594   32.204208
std      0.486592    0.836071   14.526497    1.102743    0.806057   49.693429
min      0.000000    1.000000    0.420000    0.000000    0.000000    0.000000
25%      0.000000    2.000000   20.125000    0.000000    0.000000    7.910400
50%      0.000000    3.000000   28.000000    0.000000    0.000000

### Q1(c): Missing Values Analysis

Identify all columns with missing values, report count and percentage, and provide logical explanations.

In [4]:
# Create a comprehensive missing values report
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

# Filter only columns with missing values
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Values Summary:")
print(missing_data)
print("\n" + "="*100 + "\n")

# Detailed explanation for each missing column
explanations = {
    'age': 'Age information is not available for all passengers (~20% missing). This could be due to incomplete passenger manifests or records not being digitized.',
    'deck': 'Cabin deck information is missing for ~77% of passengers. Only first-class cabins were well-documented; most second and third-class cabins lacked sufficient data.',
    'embarked': 'Port of embarkation missing for 2 passengers (~0.2%). This is likely a data entry error or incomplete record for those specific passengers.',
    'embark_town': 'Missing for the same 2 passengers as "embarked". This is derived from embarked, so the missingness is directly correlated.'
}

print("Logical Explanations for Missing Values:\n")
for col in missing_data['Column']:
    if col in explanations:
        print(f"• {col.upper()}: {explanations[col]}\n")

Missing Values Summary:
                  Column  Missing_Count  Missing_Percentage
deck                deck            688               77.22
age                  age            177               19.87
embarked        embarked              2                0.22
embark_town  embark_town              2                0.22


Logical Explanations for Missing Values:

• DECK: Cabin deck information is missing for ~77% of passengers. Only first-class cabins were well-documented; most second and third-class cabins lacked sufficient data.

• AGE: Age information is not available for all passengers (~20% missing). This could be due to incomplete passenger manifests or records not being digitized.

• EMBARKED: Port of embarkation missing for 2 passengers (~0.2%). This is likely a data entry error or incomplete record for those specific passengers.

• EMBARK_TOWN: Missing for the same 2 passengers as "embarked". This is derived from embarked, so the missingness is directly correlated.



### Q1(d): Survival Rate Analysis

Compute overall survival rate and survival rate by passenger class.

In [5]:
# Overall survival rate
overall_survived = df['survived'].sum()
overall_total = len(df)
overall_survival_rate = (overall_survived / overall_total * 100).round(2)

print("Overall Survival Statistics:")
print(f"Total Passengers: {overall_total}")
print(f"Survived: {overall_survived}")
print(f"Did Not Survive: {overall_total - overall_survived}")
print(f"Overall Survival Rate: {overall_survival_rate}%")
print("\n" + "="*100 + "\n")

# Survival rate by passenger class
survival_by_class = df.groupby('pclass').agg({
    'survived': ['count', 'sum']
}).round(2)

# Flatten column names for clarity
survival_by_class.columns = ['Total_Passengers', 'Survived']
survival_by_class['Survival_Rate_%'] = (survival_by_class['Survived'] / survival_by_class['Total_Passengers'] * 100).round(2)
survival_by_class['Did_Not_Survive'] = survival_by_class['Total_Passengers'] - survival_by_class['Survived']

# Reorder columns
survival_by_class = survival_by_class[['Total_Passengers', 'Survived', 'Did_Not_Survive', 'Survival_Rate_%']]

# Rename index for clarity
survival_by_class.index.name = 'Passenger_Class'
survival_by_class.index = ['1st Class', '2nd Class', '3rd Class']

print("Survival Rate by Passenger Class:")
print(survival_by_class)
print("\n")

# Create a comprehensive results DataFrame
results_df = pd.DataFrame({
    'Category': ['Overall'] + list(survival_by_class.index),
    'Total_Passengers': [overall_total] + survival_by_class['Total_Passengers'].tolist(),
    'Survived': [overall_survived] + survival_by_class['Survived'].tolist(),
    'Did_Not_Survive': [overall_total - overall_survived] + survival_by_class['Did_Not_Survive'].tolist(),
    'Survival_Rate_%': [overall_survival_rate] + survival_by_class['Survival_Rate_%'].tolist()
})

print("Survival Rate Summary (Formatted DataFrame):")
print(results_df.to_string(index=False))

Overall Survival Statistics:
Total Passengers: 891
Survived: 342
Did Not Survive: 549
Overall Survival Rate: 38.38%


Survival Rate by Passenger Class:
           Total_Passengers  Survived  Did_Not_Survive  Survival_Rate_%
1st Class               216       136               80            62.96
2nd Class               184        87               97            47.28
3rd Class               491       119              372            24.24


Survival Rate Summary (Formatted DataFrame):
 Category  Total_Passengers  Survived  Did_Not_Survive  Survival_Rate_%
  Overall               891       342              549            38.38
1st Class               216       136               80            62.96
2nd Class               184        87               97            47.28
3rd Class               491       119              372            24.24


---

## Q2: Data Cleaning & Feature Engineering

### Q2(a): Impute Missing Age Values

Strategy: Group-based median imputation by sex and pclass

**Justification:** Age varies significantly by gender and social class. Using a group-based median (by sex + pclass) preserves these demographic patterns better than a global median. This approach recognizes that 1st-class passengers may have had different age distributions than 3rd-class passengers, and similarly for males vs females.

**Potential Limitations:** This method assumes that missing ages within each demographic group follow similar distributions to observed values in that group. If missingness is non-random (e.g., certain age groups were less likely to be recorded), this could introduce bias.

In [6]:
# Check missing age values before imputation
print(f"Missing age values before imputation: {df['age'].isnull().sum()}")
print(f"Missing age percentage: {df['age'].isnull().sum() / len(df) * 100:.2f}%")
print("\n")

# Calculate group-based median age (by sex and pclass)
print("Median age by sex and passenger class:")
age_median_by_group = df.groupby(['sex', 'pclass'])['age'].median()
print(age_median_by_group)
print("\n")

# Perform group-based median imputation
df['age'] = df.groupby(['sex', 'pclass'])['age'].transform(lambda x: x.fillna(x.median()))

# Verify imputation
print(f"Missing age values after imputation: {df['age'].isnull().sum()}")
print("\nAge statistics after imputation:")
print(df['age'].describe())

Missing age values before imputation: 177
Missing age percentage: 19.87%


Median age by sex and passenger class:
sex     pclass
female  1         35.0
        2         28.0
        3         21.5
male    1         40.0
        2         30.0
        3         25.0
Name: age, dtype: float64


Missing age values after imputation: 0

Age statistics after imputation:
count    891.000000
mean      29.112424
std       13.304424
min        0.420000
25%       21.500000
50%       26.000000
75%       36.000000
max       80.000000
Name: age, dtype: float64


### Q2(b): Drop the Deck Column

**Justification:** 
- The `deck` column is missing for ~77% of the dataset, making it unreliable for analysis
- Imputation for categorical cabin deck would be speculative and likely inaccurate
- The information is already captured in `pclass` (which correlates with deck location)
- Dropping it preserves data quality over quantity
- We retain the `class` and `embark_town` columns which provide more complete and useful information

In [7]:
# Drop the deck column
print(f"Columns before dropping 'deck': {list(df.columns)}")
print(f"Number of columns before: {len(df.columns)}")
print("\n")

df = df.drop('deck', axis=1)

print(f"Columns after dropping 'deck': {list(df.columns)}")
print(f"Number of columns after: {len(df.columns)}")
print(f"\nVerified: 'deck' column has been removed")

Columns before dropping 'deck': ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Number of columns before: 15


Columns after dropping 'deck': ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive', 'alone']
Number of columns after: 14

Verified: 'deck' column has been removed


### Q2(c): Handle Missing Embarked Values

Use mode (most common port) to fill the 2 missing values in the embarked column.

In [8]:
# Check missing values in embarked before imputation
print(f"Missing 'embarked' values before imputation: {df['embarked'].isnull().sum()}")
print(f"Missing 'embark_town' values before imputation: {df['embark_town'].isnull().sum()}")
print("\n")

# Find the mode (most common port)
embarked_mode = df['embarked'].mode()[0]
embark_town_mode = df['embark_town'].mode()[0]

print(f"Mode of 'embarked': {embarked_mode}")
print(f"Mode of 'embark_town': {embark_town_mode}")
print("\n")

# Fill missing values with mode
df['embarked'] = df['embarked'].fillna(embarked_mode)
df['embark_town'] = df['embark_town'].fillna(embark_town_mode)

# Verify no nulls remain in these columns
print(f"Missing 'embarked' values after imputation: {df['embarked'].isnull().sum()}")
print(f"Missing 'embark_town' values after imputation: {df['embark_town'].isnull().sum()}")
print("\nVerified: No missing values in 'embarked' or 'embark_town' columns")

Missing 'embarked' values before imputation: 2
Missing 'embark_town' values before imputation: 2


Mode of 'embarked': S
Mode of 'embark_town': Southampton


Missing 'embarked' values after imputation: 0
Missing 'embark_town' values after imputation: 0

Verified: No missing values in 'embarked' or 'embark_town' columns


### Q2(d): Create Family Size and Travel Group Features

Create `family_size` column as the sum of siblings/spouses, parents/children, plus 1 (the passenger themselves).
Then create a categorical `travel_group` column based on family size.

In [9]:
# Create family_size column
df['family_size'] = df['sibsp'] + df['parch'] + 1

print("Family size distribution:")
print(df['family_size'].value_counts().sort_index())
print("\n")

# Create travel_group categorical column
def categorize_travel_group(family_size):
    if family_size == 1:
        return 'Solo'
    elif 2 <= family_size <= 4:
        return 'Small'
    else:  # family_size >= 5
        return 'Large'

df['travel_group'] = df['family_size'].apply(categorize_travel_group)

print("Travel Group value counts:")
travel_group_counts = df['travel_group'].value_counts()
print(travel_group_counts)
print("\n")

# Show the distribution for better understanding
print("Travel Group distribution (percentage):")
print(df['travel_group'].value_counts(normalize=True).mul(100).round(2))

Family size distribution:
family_size
1     537
2     161
3     102
4      29
5      15
6      22
7      12
8       6
11      7
Name: count, dtype: int64


Travel Group value counts:
travel_group
Solo     537
Small    292
Large     62
Name: count, dtype: int64


Travel Group distribution (percentage):
travel_group
Solo     60.27
Small    32.77
Large     6.96
Name: proportion, dtype: float64


### Q2(e): Create Age Group Feature

Bin ages into categories: 'Child' (0-12), 'Teen' (13-17), 'Adult' (18-59), 'Senior' (60+).
All age values have been imputed, so there should be no NaN values in the age_group.

In [10]:
# Verify no missing age values before binning
print(f"Missing age values before creating age_group: {df['age'].isnull().sum()}")
print("\n")

# Define bins and labels for age groups
bins = [0, 12, 17, 59, float('inf')]
labels = ['Child', 'Teen', 'Adult', 'Senior']

# Create age_group column using pd.cut
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

# Verify no NaN values in age_group
print(f"Missing values in age_group: {df['age_group'].isnull().sum()}")
print("\n")

# Show age_group value counts
print("Age Group distribution:")
age_group_counts = df['age_group'].value_counts().sort_index()
print(age_group_counts)
print("\n")

# Show statistics about age bins
print("Age Group statistics:")
for label in labels:
    group_data = df[df['age_group'] == label]['age']
    print(f"{label}: min={group_data.min():.1f}, max={group_data.max():.1f}, mean={group_data.mean():.1f}, count={len(group_data)}")

Missing age values before creating age_group: 0


Missing values in age_group: 0


Age Group distribution:
age_group
Child      69
Teen       44
Adult     752
Senior     26
Name: count, dtype: int64


Age Group statistics:
Child: min=0.4, max=12.0, mean=4.8, count=69
Teen: min=13.0, max=17.0, mean=15.7, count=44
Adult: min=18.0, max=59.0, mean=30.9, count=752
Senior: min=60.0, max=80.0, mean=65.1, count=26


### Q2(f): Final Null Values Check

Confirm that the cleaned dataframe has zero null values in all columns used for analysis.

In [11]:
# Check for null values across the entire dataframe
null_check = pd.DataFrame({
    'Column': df.columns,
    'Null_Count': df.isnull().sum(),
    'Null_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

print("Final Null Values Check - All Columns:")
print(null_check)
print("\n" + "="*100 + "\n")

# Count total nulls
total_nulls = df.isnull().sum().sum()
print(f"Total null values in the cleaned dataframe: {total_nulls}")
print(f"Status: {'✓ PASSED - No null values' if total_nulls == 0 else '✗ FAILED - Null values remain'}")
print("\n")

# Display final dataframe shape and info
print("Final Cleaned Dataframe Summary:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\n")

# Show a few samples of the cleaned data
print("Sample of cleaned data (first 5 rows):")
print(df.head().to_string())

Final Null Values Check - All Columns:
                    Column  Null_Count  Null_Percentage
survived          survived           0              0.0
pclass              pclass           0              0.0
sex                    sex           0              0.0
age                    age           0              0.0
sibsp                sibsp           0              0.0
parch                parch           0              0.0
fare                  fare           0              0.0
embarked          embarked           0              0.0
class                class           0              0.0
who                    who           0              0.0
adult_male      adult_male           0              0.0
embark_town    embark_town           0              0.0
alive                alive           0              0.0
alone                alone           0              0.0
family_size    family_size           0              0.0
travel_group  travel_group           0              0.0
age_group

---

## Summary: Part 1 Exploratory Data Analysis - Complete

### Data Cleaning Summary

| Step | Action | Result |
|------|--------|--------|
| Age Imputation | Group-based median (by sex + pclass) | 0 missing values |
| Deck Column | Dropped (77% missing) | Column removed |
| Embarked | Mode imputation | 0 missing values |
| Embark Town | Mode imputation | 0 missing values |
| Feature Engineering | Created family_size, travel_group, age_group | 3 new features |

### Key Findings

1. **Overall Survival Rate**: ~38.38% of passengers survived
2. **Survival by Class**:
   - 1st Class: ~62.97% (highest survival)
   - 2nd Class: ~47.28%
   - 3rd Class: ~24.24% (lowest survival)

3. **Travel Group Distribution**:
   - Solo travelers: Most common
   - Small families (2-4): Significant portion
   - Large families (5+): Few passengers

4. **Age Distribution** (after imputation):
   - Children (0-12): Young passengers
   - Teens (13-17): Adolescents
   - Adults (18-59): Main demographic
   - Seniors (60+): Elderly passengers

### Next Steps

The cleaned dataframe is ready for visualization and advanced analysis in subsequent parts of the assignment. All null values have been handled, and new features are ready for use in exploratory visualizations.